In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder\
    .appName("Module_11")\
    .master("local[*]")\
    .getOrCreate()

In [4]:
spark

# 1. Basic DataFrame Operations 


## 1. Load the sales.csv and customer.csv files into separate DataFrames. 


In [5]:
sales_df = spark.read\
    .option("header",True)\
    .csv(r"C:\Users\aman.rajput\Downloads\Module_11_Assignment\sales_dirty.csv")

In [6]:
customer_df = spark.read\
    .option("header",True)\
    .csv(r"C:\Users\aman.rajput\Downloads\Module_11_Assignment\customers_dirty.csv")

## 2. Display the schema of both DataFrames. 


In [7]:
sales_df.printSchema()

root
 |-- sales_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- region: string (nullable = true)



In [8]:
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)



## 3. Show the first 5 rows from the sales DataFrame. 


In [9]:
sales_df.show(5)

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|
|       4|       6125| Mobile|55543.0|2024-12-15| North|
|       5|       9800| Laptop|48545.0|2024-05-11|  East|
+--------+-----------+-------+-------+----------+------+
only showing top 5 rows


In [10]:
customer_df.show(5)

+-----------+-----------------+--------------------+---+---------------+
|customer_id|    customer_name|               email|age|           city|
+-----------+-----------------+--------------------+---+---------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|
|        103|  Karen Hernandez|                NULL| 28| South Courtney|
|        104| Kathleen Chapman|jessejones@exampl...| 45|    South Paige|
+-----------+-----------------+--------------------+---+---------------+
only showing top 5 rows


## 4. Count the number of rows and columns in the customer DataFrame. 


In [11]:
print("Number of Rows in sales =",sales_df.count())
print("Number of Columns in sales =",len(sales_df.columns))

Number of Rows in sales = 110000
Number of Columns in sales = 6


In [12]:
print("Number of Rows in sales =",customer_df.count())
print("Number of Columns in sales =",len(customer_df.columns))

Number of Rows in sales = 11000
Number of Columns in sales = 5


# 2. Data Cleaning 

## 5. Remove duplicate rows from the sales DataFrame based on customer_id,product,amount,sale_date,region columns

In [13]:
sales_df.show(3)

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|       1|       4296| Laptop|10299.0|2025-04-08| South|
|       2|       6307| Laptop|16842.0|2025-11-25|  East|
|       3|       5369|Desktop|64411.0|2025-04-13|  East|
+--------+-----------+-------+-------+----------+------+
only showing top 3 rows


In [14]:
sales_df = sales_df.drop_duplicates(["customer_id","product","amount","sale_date","region"])


## 6. Drop rows where any column in the customer DataFrame has null values.

In [15]:
customer_df = customer_df.dropna(how="any")

## 7. Replace null values in the amount column of the sales DataFrame with 0. 

In [16]:
sales_df.show(3)

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|   72999|        134|Desktop|  NULL|2024-07-08|  East|
|   47174|        145|Desktop|  NULL|2024-10-14|  East|
|   31014|       1546|Desktop|  NULL|2026-02-12|  East|
+--------+-----------+-------+------+----------+------+
only showing top 3 rows


In [17]:
sales_df = sales_df.fillna("0","amount")

In [18]:
sales_df.show(3)

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|   72999|        134|Desktop|     0|2024-07-08|  East|
|   47174|        145|Desktop|     0|2024-10-14|  East|
|   31014|       1546|Desktop|     0|2026-02-12|  East|
+--------+-----------+-------+------+----------+------+
only showing top 3 rows


## 8. Replace null values in the email column of the customer DataFrame with the value "unknown".

**This question contradicts the question above the previous question** <br>
**If we have dropped all the nulls values how would we fill it !!!** <br>

# 3. Column Manipulation

**Before Manipulation lets fix the schema first**

In [72]:
sales_df.show(3)

+--------+-----------+-------+-------+----------+------+
|sales_id|customer_id|product| amount| sale_date|region|
+--------+-----------+-------+-------+----------+------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|
+--------+-----------+-------+-------+----------+------+
only showing top 3 rows


In [73]:
sales_df.printSchema()

root
 |-- sales_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)



In [74]:
sales_df = sales_df.withColumn("sales_id",col("sales_id").cast("int"))\
        .withColumn("customer_id",col("customer_id").cast("int"))\
        .withColumn("amount",col("amount").cast("float"))\
        .withColumn("sale_date",to_date("sale_date"))\
        .filter(col("amount") > 0)

In [75]:
sales_df.collect()

[Row(sales_id=10778, customer_id=799, product='Laptop', amount=10004.0, sale_date=datetime.date(2025, 2, 15), region='North'),
 Row(sales_id=29443, customer_id=8499, product='Laptop', amount=10004.0, sale_date=datetime.date(2026, 3, 25), region='South'),
 Row(sales_id=55094, customer_id=2509, product='Laptop', amount=10016.0, sale_date=datetime.date(2024, 7, 15), region='East'),
 Row(sales_id=81074, customer_id=3942, product='Desktop', amount=10017.0, sale_date=datetime.date(2026, 2, 16), region='South'),
 Row(sales_id=51462, customer_id=8932, product='Laptop', amount=10020.0, sale_date=datetime.date(2025, 1, 8), region='South'),
 Row(sales_id=48401, customer_id=3164, product='Laptop', amount=10027.0, sale_date=datetime.date(2024, 4, 18), region='West'),
 Row(sales_id=68807, customer_id=8564, product='Laptop', amount=10028.0, sale_date=datetime.date(2025, 8, 9), region='South'),
 Row(sales_id=60673, customer_id=8341, product='Tablet', amount=10031.0, sale_date=datetime.date(2025, 6, 1)

In [76]:
sales_df_schema = StructType([
    StructField("sales_id",IntegerType(),False),
    StructField("customer_id",IntegerType(),False),
    StructField("product",StringType(),False),
    StructField("amount",FloatType(),False),
    StructField("sale_date",DateType(),False),
    StructField("region",StringType(),False)
])

In [77]:
cleaned_sales_df = spark.createDataFrame(sales_df.rdd,sales_df_schema)

In [78]:
cleaned_sales_df.collect()

[Row(sales_id=10778, customer_id=799, product='Laptop', amount=10004.0, sale_date=datetime.date(2025, 2, 15), region='North'),
 Row(sales_id=29443, customer_id=8499, product='Laptop', amount=10004.0, sale_date=datetime.date(2026, 3, 25), region='South'),
 Row(sales_id=55094, customer_id=2509, product='Laptop', amount=10016.0, sale_date=datetime.date(2024, 7, 15), region='East'),
 Row(sales_id=81074, customer_id=3942, product='Desktop', amount=10017.0, sale_date=datetime.date(2026, 2, 16), region='South'),
 Row(sales_id=51462, customer_id=8932, product='Laptop', amount=10020.0, sale_date=datetime.date(2025, 1, 8), region='South'),
 Row(sales_id=48401, customer_id=3164, product='Laptop', amount=10027.0, sale_date=datetime.date(2024, 4, 18), region='West'),
 Row(sales_id=68807, customer_id=8564, product='Laptop', amount=10028.0, sale_date=datetime.date(2025, 8, 9), region='South'),
 Row(sales_id=60673, customer_id=8341, product='Tablet', amount=10031.0, sale_date=datetime.date(2025, 6, 1)

In [79]:
cleaned_sales_df.printSchema()

root
 |-- sales_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- product: string (nullable = false)
 |-- amount: float (nullable = false)
 |-- sale_date: date (nullable = false)
 |-- region: string (nullable = false)



In [80]:
customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+
|customer_id|    customer_name|               email|age|           city|
+-----------+-----------------+--------------------+---+---------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|
+-----------+-----------------+--------------------+---+---------------+
only showing top 3 rows


In [81]:
customer_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)



In [82]:
customer_df = customer_df.withColumn("customer_id",col("customer_id").cast("int"))\
            .withColumn("age",col("age").cast("int"))

In [83]:
customer_df.collect()

[Row(customer_id=100, customer_name='Jasmine Contreras', email='xmacias@example.org', age=49, city='Hollandtown'),
 Row(customer_id=101, customer_name='Michael Jones', email='michaelkhan@example.org', age=27, city='New Kennethstad'),
 Row(customer_id=102, customer_name='Dr. Jason Murray', email='penapatricia@example.net', age=37, city='West Megan'),
 Row(customer_id=104, customer_name='Kathleen Chapman', email='jessejones@example.org', age=45, city='South Paige'),
 Row(customer_id=105, customer_name='Melissa Greene', email='deanaustin@example.com', age=24, city='Charlesberg'),
 Row(customer_id=107, customer_name='Dr. Paul Bautista', email='zbrennan@example.com', age=37, city='North Kayla'),
 Row(customer_id=108, customer_name='Samuel Garza', email='edward54@example.net', age=33, city='East Michael'),
 Row(customer_id=109, customer_name='Lori Morris', email='pricesusan@example.net', age=46, city='New Jamesview'),
 Row(customer_id=110, customer_name='Jennifer Black', email='pjordan@examp

In [84]:
customer_df_schema = StructType([
    StructField("customer_id",IntegerType(),False),
    StructField("customer_name",StringType(),False),
    StructField("email",StringType(),False),
    StructField("age",IntegerType(),False),
    StructField("city",StringType(),False)
])

In [85]:
cleaned_customer_df = spark.createDataFrame(customer_df.rdd,customer_df_schema)

In [86]:
cleaned_customer_df.show()

+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|             city|
+-----------+------------------+--------------------+---+-----------------+
|        100| Jasmine Contreras| xmacias@example.org| 49|      Hollandtown|
|        101|     Michael Jones|michaelkhan@examp...| 27|  New Kennethstad|
|        102|  Dr. Jason Murray|penapatricia@exam...| 37|       West Megan|
|        104|  Kathleen Chapman|jessejones@exampl...| 45|      South Paige|
|        105|    Melissa Greene|deanaustin@exampl...| 24|      Charlesberg|
|        107| Dr. Paul Bautista|zbrennan@example.com| 37|      North Kayla|
|        108|      Samuel Garza|edward54@example.net| 33|     East Michael|
|        109|       Lori Morris|pricesusan@exampl...| 46|    New Jamesview|
|        110|    Jennifer Black| pjordan@example.com| 69|      Lynnchester|
|        111|Melissa Richardson|  marias@example.net| 57|   New Staceyfort|
|        112

In [87]:
cleaned_customer_df.printSchema()

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- city: string (nullable = false)



In [88]:
cleaned_customer_df.collect()

[Row(customer_id=100, customer_name='Jasmine Contreras', email='xmacias@example.org', age=49, city='Hollandtown'),
 Row(customer_id=101, customer_name='Michael Jones', email='michaelkhan@example.org', age=27, city='New Kennethstad'),
 Row(customer_id=102, customer_name='Dr. Jason Murray', email='penapatricia@example.net', age=37, city='West Megan'),
 Row(customer_id=104, customer_name='Kathleen Chapman', email='jessejones@example.org', age=45, city='South Paige'),
 Row(customer_id=105, customer_name='Melissa Greene', email='deanaustin@example.com', age=24, city='Charlesberg'),
 Row(customer_id=107, customer_name='Dr. Paul Bautista', email='zbrennan@example.com', age=37, city='North Kayla'),
 Row(customer_id=108, customer_name='Samuel Garza', email='edward54@example.net', age=33, city='East Michael'),
 Row(customer_id=109, customer_name='Lori Morris', email='pricesusan@example.net', age=46, city='New Jamesview'),
 Row(customer_id=110, customer_name='Jennifer Black', email='pjordan@examp

## 9. Add a new column discounted_amount to the sales DataFrame that applies a 10% discount on amount. 

In [89]:
cleaned_sales_df = cleaned_sales_df.withColumn("discounted_amount",round(col("amount")*0.9,2))

In [90]:
cleaned_sales_df.show(4)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
|   81074|       3942|Desktop|10017.0|2026-02-16| South|           9015.3|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 4 rows


## 10. Rename the city column in the customer DataFrame to customer_city. 

In [91]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+
|customer_id|    customer_name|               email|age|           city|
+-----------+-----------------+--------------------+---+---------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|
+-----------+-----------------+--------------------+---+---------------+
only showing top 3 rows


In [92]:
cleaned_customer_df = cleaned_customer_df.withColumnRenamed("city","customer_city")

In [94]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+
|customer_id|    customer_name|               email|age|  customer_city|
+-----------+-----------------+--------------------+---+---------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|
+-----------+-----------------+--------------------+---+---------------+
only showing top 3 rows


## 11. Drop the region column from the sales DataFrame. 

**Not dropping region column because it is used in the 32nd question**

## 12. Create a new column customer_age_category in the customer DataFrame based on age: 
- "Youth" for age < 30 
- "Adult" for 30 <= age < 50 
- "Senior" for age >= 50 


In [96]:
cleaned_customer_df = cleaned_customer_df.withColumn(
    "customer_age_category",
    when(col("age") >= 50, 'Senior')
    .when((col("age") >= 30) & (col("age") < 50),'Adult')
    .otherwise("Youth")
    )

In [97]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+---------------------+
|customer_id|    customer_name|               email|age|  customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+---------------+---------------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|                Adult|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|                Adult|
+-----------+-----------------+--------------------+---+---------------+---------------------+
only showing top 3 rows


# 4. Filtering 

## 13. Filter the sales DataFrame to show only rows where amount is greater than 50,000. 

In [103]:
cleaned_sales_df.filter(col("amount") > 50000).show()

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   47916|       3270| Laptop|50002.0|2025-06-16|  East|          45001.8|
|   51105|       5871| Laptop|50003.0|2025-07-27| North|          45002.7|
|   60714|       6206| Mobile|50003.0|2026-02-27| North|          45002.7|
|   18235|        436| Tablet|50004.0|2026-02-08| North|          45003.6|
|    3557|        946| Mobile|50004.0|2024-12-14| South|          45003.6|
|   83792|       1061| Laptop|50006.0|2024-04-17|  East|          45005.4|
|   44699|       3276| Laptop|50007.0|2024-10-30| South|          45006.3|
|   40529|       1789|Desktop|50011.0|2026-01-24| South|          45009.9|
|    4060|       6973| Laptop|50014.0|2025-09-16|  West|          45012.6|
|   45037|       3159|Desktop|50015.0|2025-10-08| North|          45013.5|
|   84099|       5268| Mo

## 14. Filter the customer DataFrame to show customers aged between 25 and 30.

In [105]:
cleaned_customer_df.filter( (col("age")>=25) & (col("age")<=30) ).show()

+-----------+-----------------+--------------------+---+------------------+---------------------+
|customer_id|    customer_name|               email|age|     customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+------------------+---------------------+
|        101|    Michael Jones|michaelkhan@examp...| 27|   New Kennethstad|                Youth|
|        138|    Chelsea Ortiz|kellycollins@exam...| 29|         Emilyview|                Youth|
|        140|      Eric Cortez|elaine06@example.org| 29|         Meganfurt|                Youth|
|        153|   Timothy Miller|smithjack@example...| 26|        Conniestad|                Youth|
|        162|Justin Coleman MD|  john68@example.com| 28|        Pamelastad|                Youth|
|        171|  Brooke Gonzales| james62@example.net| 29|        Lake Jacob|                Youth|
|        217|    Tiffany Lopez| james17@example.com| 26|        Judithbury|                Youth|
|        222|     Ma

## 15. Identify all customers who have made purchases in more than one region. 

In [114]:

from pyspark.sql.functions import countDistinct


In [117]:
cleaned_sales_df.groupby("customer_id").agg(
    countDistinct("region").alias("count_distinct_region")
).filter(col("count_distinct_region") > 1).sort("customer_id").show()

+-----------+---------------------+
|customer_id|count_distinct_region|
+-----------+---------------------+
|        100|                    4|
|        101|                    4|
|        102|                    4|
|        103|                    3|
|        104|                    3|
|        105|                    2|
|        106|                    2|
|        107|                    3|
|        108|                    4|
|        109|                    4|
|        110|                    4|
|        111|                    2|
|        112|                    4|
|        113|                    4|
|        114|                    4|
|        115|                    3|
|        116|                    4|
|        117|                    4|
|        118|                    3|
|        119|                    4|
+-----------+---------------------+
only showing top 20 rows


## 16. Filter the top 3 sales based on amount for each product. 

In [118]:
cleaned_sales_df.show()

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
|   81074|       3942|Desktop|10017.0|2026-02-16| South|           9015.3|
|   51462|       8932| Laptop|10020.0|2025-01-08| South|           9018.0|
|   48401|       3164| Laptop|10027.0|2024-04-18|  West|           9024.3|
|   68807|       8564| Laptop|10028.0|2025-08-09| South|           9025.2|
|   60673|       8341| Tablet|10031.0|2025-06-01| North|           9027.9|
|   41676|       4712| Tablet|10037.0|2025-08-26|  East|           9033.3|
|   28560|       5416| Tablet|10042.0|2025-01-06| South|           9037.8|
|   77060|       3584|Des

In [119]:
from pyspark.sql.window import Window

In [120]:
windowSpec = Window.partitionBy("product").orderBy(col("amount").desc())

In [124]:
cleaned_sales_df.withColumn("dense_rank",dense_rank().over(windowSpec)).filter(col("dense_rank") <= 3).show()

+--------+-----------+-------+-------+----------+------+-----------------+----------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|dense_rank|
+--------+-----------+-------+-------+----------+------+-----------------+----------+
|   20093|       6368|Desktop|99992.0|2025-06-13|  East|          89992.8|         1|
|   28608|       3517|Desktop|99985.0|2024-05-19| North|          89986.5|         2|
|    3913|       2539|Desktop|99968.0|2025-03-04| South|          89971.2|         3|
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|         1|
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|         1|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|         2|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|         3|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|         3|
|   37866|        636| Mobile|99995.0|2025-05-07|  Wes

# 5. Joins 

In [125]:
print(cleaned_sales_df.show(3))

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows
None


In [126]:
print(cleaned_customer_df.show(3))

+-----------+-----------------+--------------------+---+---------------+---------------------+
|customer_id|    customer_name|               email|age|  customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+---------------+---------------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|                Adult|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|                Adult|
+-----------+-----------------+--------------------+---+---------------+---------------------+
only showing top 3 rows
None


## 17. Perform an inner join between sales and customer DataFrames on customer_id. 

In [129]:
cleaned_sales_df.join(cleaned_customer_df,"customer_id","inner").show(3)

+-----------+--------+-------+-------+----------+------+-----------------+-----------------+--------------------+---+-------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region|discounted_amount|    customer_name|               email|age|customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+-----------------+-----------------+--------------------+---+-------------+---------------------+
|       7340|   84330| Laptop|10411.0|2024-09-07| North|           9369.9|Michelle Schaefer|parksamuel@exampl...| 37| Tiffanyshire|                Adult|
|       7340|   41344| Laptop|10545.0|2025-01-18| North|           9490.5|Michelle Schaefer|parksamuel@exampl...| 37| Tiffanyshire|                Adult|
|       8592|   97545| Mobile|11905.0|2025-12-02|  East|          10714.5|  Jordan Martinez|jamesforbes@examp...| 19|    Lake Sean|                Youth|
+-----------+--------+-------+-------+----------+------+-----------------+--

## 18. Perform a left join to include all records from sales and matching records from customer.

In [130]:
cleaned_sales_df.join(cleaned_customer_df,"customer_id","left").show(3)

+-----------+--------+-------+-------+----------+------+-----------------+------------------+--------------------+---+-------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region|discounted_amount|     customer_name|               email|age|customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+-----------------+------------------+--------------------+---+-------------+---------------------+
|        799|   10778| Laptop|10004.0|2025-02-15| North|           9003.6|Richard Richardson| janet14@example.org| 45|Nathanchester|                Adult|
|       8499|   29443| Laptop|10004.0|2026-03-25| South|           9003.6|        Scott Mann|belindahiggins@ex...| 59|    Tranhaven|               Senior|
|       3942|   81074|Desktop|10017.0|2026-02-16| South|           9015.3|       David Lucas|  alec04@example.org| 48|    Lynchbury|                Adult|
+-----------+--------+-------+-------+----------+------+--------------

## 19. Perform a full outer join between sales and customer DataFrames. 

In [131]:
cleaned_sales_df.join(cleaned_customer_df,"customer_id","outer").show(3)

+-----------+--------+-------+-------+----------+------+-----------------+-------------+--------------------+---+---------------+---------------------+
|customer_id|sales_id|product| amount| sale_date|region|discounted_amount|customer_name|               email|age|  customer_city|customer_age_category|
+-----------+--------+-------+-------+----------+------+-----------------+-------------+--------------------+---+---------------+---------------------+
|        101|   74001| Tablet|94761.0|2025-11-08|  East|          85284.9|Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        101|   72057| Tablet|22021.0|2024-11-04| North|          19818.9|Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        101|   69645|Desktop|23963.0|2026-03-14| North|          21566.7|Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
+-----------+--------+-------+-------+----------+------+-----------------+-------------+

## 20. Identify customers who have not made any purchases by performing an anti-join. 

In [132]:
cleaned_customer_df.join(cleaned_sales_df,"customer_id","left_anti").show(3)

+-----------+-------------+-----+---+-------------+---------------------+
|customer_id|customer_name|email|age|customer_city|customer_age_category|
+-----------+-------------+-----+---+-------------+---------------------+
+-----------+-------------+-----+---+-------------+---------------------+



# 6. Aggregations 

## 21. Calculate the total sales amount for each product. 

In [133]:
cleaned_sales_df.show(3)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows


In [136]:
cleaned_sales_df.groupby("product").agg(sum("amount").alias("total_sales_amount")).show()

+-------+------------------+
|product|total_sales_amount|
+-------+------------------+
| Laptop|     3.066342959E9|
| Mobile|      6.06916252E8|
| Tablet|      6.19847504E8|
|Desktop|      6.05374242E8|
+-------+------------------+



## 22. Find the average age of customers in the customer DataFrame.

In [137]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+---------------------+
|customer_id|    customer_name|               email|age|  customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+---------------+---------------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|                Adult|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|                Adult|
+-----------+-----------------+--------------------+---+---------------+---------------------+
only showing top 3 rows


In [140]:
cleaned_customer_df.agg(round(avg("age"),0).alias("Average_Age")).show()

+-----------+
|Average_Age|
+-----------+
|       44.0|
+-----------+



## 23. Calculate the maximum and minimum sales amounts in the sales DataFrame.

In [141]:
cleaned_sales_df.agg(
    max("amount").alias("max_sales_amount"),
    min("amount").alias("min_sales_amount")
).show()

+----------------+----------------+
|max_sales_amount|min_sales_amount|
+----------------+----------------+
|         99999.0|          5001.0|
+----------------+----------------+



## 24. Group the customer DataFrame by customer_city and count the number of customers in each city.

In [142]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+---------------------+
|customer_id|    customer_name|               email|age|  customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+---------------+---------------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|                Adult|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|                Adult|
+-----------+-----------------+--------------------+---+---------------+---------------------+
only showing top 3 rows


In [143]:
cleaned_customer_df.groupby("customer_city").agg(
    count("customer_id").alias("Number_of_customers")
).show()

+------------------+-------------------+
|     customer_city|Number_of_customers|
+------------------+-------------------+
|         Dianaland|                  1|
|     East Amymouth|                  2|
|  South Nicoleport|                  1|
|        Denisefurt|                  1|
|   New Michelefurt|                  1|
|       Port Monica|                  1|
|          Annefurt|                  1|
|       Lake Joshua|                  7|
|     Wilsonchester|                  2|
|     New Scottstad|                  1|
|   South Amberfurt|                  2|
| New Jennifershire|                  2|
|         Austinton|                  3|
|        East Tammy|                  4|
|South Nicholastown|                  1|
|      East Jeffrey|                  2|
|          Karaland|                  1|
|         East Cory|                  1|
|        Karenmouth|                  1|
|        Lake April|                  2|
+------------------+-------------------+
only showing top

# 7. Sorting

## 25. Sort the sales DataFrame by amount in descending order. 

In [144]:
cleaned_sales_df.show(3)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows


In [150]:
cleaned_sales_df.orderBy(col("amount").desc()).show()

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|
|    6874|       6345| Tablet|99997.0|2025-06-02| South|          89997.3|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|
|   37866|        636| Mobile|99995.0|2025-05-07|  West|          89995.5|
|   71825|       5731| Mobile|99993.0|2024-06-15| South|          89993.7|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|
|   70208|       8097| Laptop|99992.0|2024-08-28|  West|          89992.8|
|   20093|       6368|Desktop|99992.0|2025-06-13|  East|          89992.8|
|   26795|       9215| La

## 26. Sort the customer DataFrame by age in ascending order. 

In [151]:
cleaned_customer_df.show(3)

+-----------+-----------------+--------------------+---+---------------+---------------------+
|customer_id|    customer_name|               email|age|  customer_city|customer_age_category|
+-----------+-----------------+--------------------+---+---------------+---------------------+
|        100|Jasmine Contreras| xmacias@example.org| 49|    Hollandtown|                Adult|
|        101|    Michael Jones|michaelkhan@examp...| 27|New Kennethstad|                Youth|
|        102| Dr. Jason Murray|penapatricia@exam...| 37|     West Megan|                Adult|
+-----------+-----------------+--------------------+---+---------------+---------------------+
only showing top 3 rows


In [152]:
cleaned_customer_df.orderBy(col("age")).show()

+-----------+-------------------+--------------------+---+------------------+---------------------+
|customer_id|      customer_name|               email|age|     customer_city|customer_age_category|
+-----------+-------------------+--------------------+---+------------------+---------------------+
|       1148|       Tammy Rogers|rachael67@example...|  5|       Joshuamouth|                Youth|
|       2983|     Dr. Paul Brown|jesusstephens@exa...|  5|South Elizabethton|                Youth|
|       1161|     Rebecca Morgan|ballardjacob@exam...|  5|      Daniellefurt|                Youth|
|        119|   Jasmine Williams|peterjohnson@exam...|  5|          Lynnstad|                Youth|
|       1172|      Michele Reese|kevinroberts@exam...|  5|      Shepherdstad|                Youth|
|        308|      Rhonda Harris|jonathan15@exampl...|  5|        West David|                Youth|
|       1265|Mr. George Saunders|santiagowendy@exa...|  5|   West Andrewland|                Youth|


# 8. Union Operations 

## 27. Add a new dataset for customers and perform a union operation with the customer DataFrame. 

In [153]:
cleaned_customer_df.printSchema()

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- customer_city: string (nullable = false)
 |-- customer_age_category: string (nullable = false)



In [154]:
from pyspark.sql import Row

In [156]:
data2 = [
    Row(customer_id=5001, customer_name="Aman Rajput",
        email="aman.rajput@example.com", age=18,
        customer_city="Newport", customer_age_category="Adult"),

    Row(customer_id=5002, customer_name="Smith Thakkar",
        email="smith.thakkar@example.com", age=12,
        customer_city="Lakeview", customer_age_category="Adult")
]

In [158]:
df2 = spark.createDataFrame(data2, schema=cleaned_customer_df.schema)

In [159]:
cleaned_customer_df = cleaned_customer_df.union(df2)

## 28. Combine the sales DataFrame with another DataFrame containing additional sales records.

In [160]:
cleaned_sales_df.show(3)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows


In [161]:
cleaned_sales_df.printSchema()

root
 |-- sales_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- product: string (nullable = false)
 |-- amount: float (nullable = false)
 |-- sale_date: date (nullable = false)
 |-- region: string (nullable = false)
 |-- discounted_amount: double (nullable = true)



In [162]:
from datetime import date

new_sales_data = [
    (70001, 1234, "Laptop", 12000.0, date(2025, 5, 10), "West", 10800.0),
    (70002, 5678, "Tablet",  8000.0, date(2025, 8, 20), "North", 7200.0)
]

In [163]:
df_new_sales = spark.createDataFrame(new_sales_data, schema=cleaned_sales_df.schema)

In [164]:
cleaned_sales_df = cleaned_sales_df.union(df_new_sales)

In [166]:
cleaned_sales_df.show(3)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows


# 9. Window Functions 

In [168]:
windowSpec = Window.partitionBy("product").orderBy(col("amount").desc())

## 29. Rank the sales records based on the amount column.

In [167]:
cleaned_sales_df.show(3)

+--------+-----------+-------+-------+----------+------+-----------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|
+--------+-----------+-------+-------+----------+------+-----------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|
+--------+-----------+-------+-------+----------+------+-----------------+
only showing top 3 rows


In [169]:
cleaned_sales_df.withColumn("rank",dense_rank().over(windowSpec)).show()

+--------+-----------+-------+-------+----------+------+-----------------+----+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|rank|
+--------+-----------+-------+-------+----------+------+-----------------+----+
|   69049|       6637| Laptop|99999.0|2025-05-10| South|          89999.1|   1|
|    3486|       1646| Laptop|99999.0|2024-12-13| South|          89999.1|   1|
|   96541|       9079| Laptop|99995.0|2024-05-26| North|          89995.5|   2|
|    8104|       6810| Laptop|99993.0|2024-06-28|  West|          89993.7|   3|
|   78912|       9982| Laptop|99993.0|2025-12-31|  West|          89993.7|   3|
|   70208|       8097| Laptop|99992.0|2024-08-28|  West|          89992.8|   4|
|   26795|       9215| Laptop|99991.0|2025-12-16| South|          89991.9|   5|
|   49812|       3406| Laptop|99987.0|2024-10-13|  East|          89988.3|   6|
|   34968|       9208| Laptop|99983.0|2025-08-16| South|          89984.7|   7|
|    6019|       3740| Laptop|99982.0|20

## 30. Add a cumulative sum of amount for each product in the sales DataFrame. 

In [172]:
windowSpec = Window.partitionBy("product").orderBy(col("sale_date")).rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [173]:
cleaned_sales_df.withColumn("cum_sales",sum("amount").over(windowSpec)).show()

+--------+-----------+-------+-------+----------+------+-----------------+---------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|cum_sales|
+--------+-----------+-------+-------+----------+------+-----------------+---------+
|   31586|        720| Laptop|15825.0|2024-03-25| South|          14242.5|  15825.0|
|   92268|       7434| Laptop|18031.0|2024-03-25|  West|          16227.9|  33856.0|
|   14642|       4518| Laptop|24718.0|2024-03-25|  West|          22246.2|  58574.0|
|   79506|       9969| Laptop|25457.0|2024-03-25|  East|          22911.3|  84031.0|
|   76187|       9593| Laptop|34423.0|2024-03-25| North|          30980.7| 118454.0|
|   75002|        372| Laptop|37830.0|2024-03-25| South|          34047.0| 156284.0|
|   58817|       1363| Laptop|38191.0|2024-03-25| North|          34371.9| 194475.0|
|    3532|       4577| Laptop|40971.0|2024-03-25|  East|          36873.9| 235446.0|
|    3388|       2649| Laptop|49811.0|2024-03-25| South|         

## 31. Add a column that calculates the difference between each customer's amount and the average amount within their product group.

In [174]:
windowSpec = Window.partitionBy("product")

In [179]:
cleaned_sales_df.withColumn("average_product_wise",avg("amount").over(windowSpec))\
    .withColumn("difference",col("amount")-col("average_product_wise")).show()

+--------+-----------+-------+-------+----------+------+-----------------+--------------------+------------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|average_product_wise|        difference|
+--------+-----------+-------+-------+----------+------+-----------------+--------------------+------------------+
|   10778|        799| Laptop|10004.0|2025-02-15| North|           9003.6|   52617.80079277919|-42613.80079277919|
|   29443|       8499| Laptop|10004.0|2026-03-25| South|           9003.6|   52617.80079277919|-42613.80079277919|
|   55094|       2509| Laptop|10016.0|2024-07-15|  East|           9014.4|   52617.80079277919|-42601.80079277919|
|   51462|       8932| Laptop|10020.0|2025-01-08| South|           9018.0|   52617.80079277919|-42597.80079277919|
|   48401|       3164| Laptop|10027.0|2024-04-18|  West|           9024.3|   52617.80079277919|-42590.80079277919|
|   68807|       8564| Laptop|10028.0|2025-08-09| South|           9025.2|   526

# 10. Partitioning

## 32. Write the sales DataFrame to a partitioned Parquet file by region.

In [184]:
cleaned_sales_df.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(r"C:\Users\aman.rajput\Downloads\Module_9_Assignment")

Py4JJavaError: An error occurred while calling o723.parquet.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.

## 33. Partition the customer DataFrame by customer_city and save it as a CSV file.

In [185]:
cleaned_customer_df.write \
    .mode("overwrite") \
    .partitionBy("customer_city") \
    .option("header", "true") \
    .csv("output/customer_by_city_csv")

Py4JJavaError: An error occurred while calling o731.csv.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawL

# 11. Real-World Scenarios 

## 34. Calculate the percentage contribution of each product to the total sales.

In [190]:
sum_value = cleaned_sales_df.agg(sum("amount")).first()[0]
print(sum_value)

4898500957.0


In [ ]:
cleaned_sales_df.show(3)

In [195]:
cleaned_sales_df.groupby("product").agg((sum("amount")*100/sum_value).alias("percentage_contribution")).show()

+-------+-----------------------+
|product|percentage_contribution|
+-------+-----------------------+
| Laptop|     62.597823005794304|
| Mobile|     12.389836346417601|
| Tablet|     12.653983523555736|
|Desktop|     12.358357124232363|
+-------+-----------------------+



## 35. Extract the year from sale_date and group by year to calculate total sales.

In [201]:
cleaned_sales_df.withColumn("year",year("sale_date"))\
                .groupby("year").agg(
                    sum("amount").alias("total_amount")
                )\
                .sort("year")\
                .show()

+----+-------------+
|year| total_amount|
+----+-------------+
|2024|1.896992085E9|
|2025|2.443442419E9|
|2026| 5.58066453E8|
+----+-------------+



# Using SQL

In [203]:
cleaned_sales_df.createOrReplaceTempView("sales")
cleaned_customer_df.createOrReplaceTempView("customers")

## 36. Identify the most purchased product in each region.

In [204]:
spark.sql("""
WITH product_counts AS (
    SELECT
        region,
        product,
        COUNT(*) AS cnt
    FROM sales
    GROUP BY region, product
),
ranked AS (
    SELECT
        region,
        product,
        cnt,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY cnt DESC) AS rn
    FROM product_counts
)
SELECT region, product AS most_purchased_product, cnt
FROM ranked
WHERE rn = 1
""").show()

+------+----------------------+-----+
|region|most_purchased_product|  cnt|
+------+----------------------+-----+
|  East|                Laptop|14445|
| North|                Laptop|14396|
| South|                Laptop|14813|
|  West|                Laptop|14622|
+------+----------------------+-----+



## 37. Add a column to show the difference between the highest and lowest sales for each product. 

In [205]:
spark.sql("""
SELECT
    product,
    MAX(amount) AS max_amount,
    MIN(amount) AS min_amount,
    MAX(amount) - MIN(amount) AS amount_difference
FROM sales
GROUP BY product
""").show()

+-------+----------+----------+-----------------+
|product|max_amount|min_amount|amount_difference|
+-------+----------+----------+-----------------+
| Laptop|   99999.0|    5001.0|          94998.0|
| Mobile|   99995.0|    5002.0|          94993.0|
| Tablet|   99997.0|    5004.0|          94993.0|
|Desktop|   99992.0|    5018.0|          94974.0|
+-------+----------+----------+-----------------+



## 38. Write the result of the join between sales and customer to parquet file. 

In [210]:
spark.sql("""
SELECT
    s.*,
    c.customer_name,
    c.email,
    c.customer_city,
    c.customer_age_category
FROM sales s
JOIN customers c
ON s.customer_id = c.customer_id
""").show()

+--------+-----------+-------+-------+----------+------+-----------------+----------------+--------------------+---------------+---------------------+
|sales_id|customer_id|product| amount| sale_date|region|discounted_amount|   customer_name|               email|  customer_city|customer_age_category|
+--------+-----------+-------+-------+----------+------+-----------------+----------------+--------------------+---------------+---------------------+
|   73519|        148| Mobile|94510.0|2025-06-22|  East|          85059.0|    Rachel Parks|  awoods@example.net|       Penatown|                Adult|
|   32839|        148| Laptop|93279.0|2024-12-28|  West|          83951.1|    Rachel Parks|  awoods@example.net|       Penatown|                Adult|
|    9402|        148| Laptop|88808.0|2024-07-09|  West|          79927.2|    Rachel Parks|  awoods@example.net|       Penatown|                Adult|
|    6964|        148|Desktop|76115.0|2025-05-02| North|          68503.5|    Rachel Parks|  a

## 39. Identify products that were sold in the last 6 months. 

In [209]:
spark.sql("""
SELECT DISTINCT product
FROM sales
WHERE sale_date >= DATE_SUB('2026-03-27', 180)
""").show()

+-------+
|product|
+-------+
| Laptop|
| Mobile|
| Tablet|
|Desktop|
+-------+



## 40. Calculate the average sales amount per customer.

In [211]:
spark.sql("""
SELECT
    customer_id,
    AVG(amount) AS average_sales_amount
FROM sales
GROUP BY customer_id
""").show()

+-----------+--------------------+
|customer_id|average_sales_amount|
+-----------+--------------------+
|       7340|             37627.5|
|       8592|           68371.875|
|       5156|   56005.42857142857|
|       2659|   49934.92307692308|
|       7982|             49428.3|
|       4900|   49032.71428571428|
|        833|   49132.92857142857|
|       6658|   43514.57142857143|
|       4519|             35932.4|
|       1645|   48199.46153846154|
|       7253|           36291.375|
|       2366|             59532.5|
|        148|             61688.5|
|       3918|   42536.78571428572|
|       3794|             45567.0|
|       6466|  32723.846153846152|
|       9900|             33808.5|
|       3749|           42564.625|
|       5803|  46762.181818181816|
|       7754|   63333.07692307692|
+-----------+--------------------+
only showing top 20 rows
